In [1]:
import pandas as pd
import duckdb as ddb
from pathlib import Path

In [3]:
def preprocess_missing_data(data, power_metric="scaphandre_vm_power_total_watts", group_col="vm_id"):

    df = data.sort_values([group_col, "timestamp"])

    df["power_filled"] = (df.groupby(group_col)[power_metric].ffill())
    
    df["power_filled"] = df["power_filled"].fillna(0)

    return df

In [4]:
def construct_vm_list(vm_df):
    VMs_per_t = {}

    for t, group in vm_df.groupby("timestamp"):
        VMs_t = []

        for _, row in group.iterrows():
            vm = {
                "vm_id": row["vm_id"],
                "node_name": row["hypervisor_name"],
                "user_id": row["user_id"],
                "project_id": row["project_id"],
                "power": row["power_filled"],
                #"vcpus": row["vcpus"],
                #"memory": row["memory_mb"],
            }
            VMs_t.append(vm)

        VMs_per_t[t] = VMs_t

    return VMs_per_t

In [7]:
def construct_host_list(nodes_df):
    host_per_t = {}

    # a dicitonary with lists
    for t, group in nodes_df.groupby("timestamp"):
        host_t = []

        for _, row in group.iterrows():
            node = {
                "node_name": row["node_name"],
                "node_group": row["node_group"],
                "cpu_usage": row["cpu_usage_percent"],
                "memory_usage": row["memory_util_ratio"],
                "disk_usage": row["disk_util_ratio"],
                "power": row["impi_power_total_watts"],
            }
            host_t.append(node)

        host_per_t[t] = host_t
    
    return host_per_t

In [2]:
BASE = Path.cwd()
DATAPATH = BASE / "datasets/cloud_energy_consumption/full_nodes_featurestwo.parquet"

In [ ]:
VM_HARDWARE = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/vms/2024-12-14T000000Z_2025-04-13T235959Z/vms.csv"
VM_DATA = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/nodes-vms/2024-12-14T000000Z_2025-04-13T235959Z/**/*.csv"

In [4]:
con = ddb.connect(':memory:')

## VM HARDWARE

In [5]:
# VM HARDWARE 
con.query(f"""CREATE OR REPLACE TABLE vmhardware AS SELECT * FROM '{VM_HARDWARE}'""")
con.query("""SELECT * FROM vmhardware LIMIT 5""").show()

┌──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┐
│  vm_id   │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │
│ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │
├──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┤
│ 6239bcf4 │ 39786247 │ 4ba6ce42   │ 85ecdff4  │   32.0 │   60000.0 │    30.0 │        160.0 │
│ 80458243 │ bd41a7e4 │ 74dfdf00   │ cb92696e  │    1.0 │    2000.0 │    10.0 │         20.0 │
│ 5d9f6237 │ 7190ddfa │ 906147a9   │ 78c5906a  │    4.0 │    7500.0 │    30.0 │         60.0 │
│ 941785dc │ 1a324c91 │ f09054da   │ 23658854  │    2.0 │    4000.0 │    10.0 │        100.0 │
│ fb856e0f │ c1eb1c1c │ f09054da   │ c015d680  │    4.0 │    7500.0 │    30.0 │         60.0 │
└──────────┴──────────┴────────────┴───────────┴────────┴───────────┴─────────┴──────────────┘



## VM DATA

In [6]:
con.execute(f"""
                CREATE OR REPLACE VIEW vm_data AS
                SELECT *
                FROM read_csv_auto('{VM_DATA}',
                                filename=false,
                                union_by_name=true)
            """)

### Merging

In [7]:
con.execute("""
    CREATE OR REPLACE TABLE vm_merged AS
    SELECT 
        d.*, 
        h.*
    FROM vm_data d
    LEFT JOIN vmhardware h
    ON d.vm_id = h.vm_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Imputing missing values
Forward Fill and 0s

In [8]:
con.execute("""
    CREATE OR REPLACE TABLE vm_filled AS
    SELECT *,
        LAST_VALUE(scaphandre_vm_power_total_watts IGNORE NULLS)
        OVER (
            PARTITION BY vm_id 
            ORDER BY timestamp
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS power_filled
    FROM vm_merged
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
con.execute("""
    CREATE OR REPLACE TABLE vm_final AS
    SELECT *,
        COALESCE(power_filled, 0) AS power_clean
    FROM vm_filled
 """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
con.query("""SELECT * FROM vm_final LIMIT 5""").show()

┌──────────────────────────┬──────────┬─────────────────┬──────────────────┬─────────────────────────────────┬──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┬──────────────┬─────────────┐
│        timestamp         │  vm_id   │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ vm_id_1  │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │ power_filled │ power_clean │
│ timestamp with time zone │ varchar  │     varchar     │     varchar      │             double              │ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │    double    │   double    │
├──────────────────────────┼──────────┼─────────────────┼──────────────────┼─────────────────────────────────┼──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┼──────────────┼─────────────┤
│ 2025-03-09 08:03:00+01   │ ed2f4471 │ 22655375        │ a6177608  

In [20]:
TIMESTAMP = '2025-01-16T08:33:00Z'

### Get VM list per node

In [27]:
# Check timestamp notation
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        LIST(vm_id) AS vm_ids
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ node_name │                                                               vm_ids                                                               │
│  varchar  │                                                             varchar[]                                                              │
├───────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 85ebbabd  │ [9679fdac, aa7cd1e3, 46e19c93, 54088b57, 8e62a0df, 6f89a069, 047c438e, 54153004]                                                   │
│ 2fded806  │ [10cc9124, da223985, 6829ff4f, e9adec9a, 62ed6b58, 154203e9, c590b2a9, 3256a83b]                                                   │
│ 8d2843f5  │ [f3aa7c8f, 1619bf22, 49f37e7d, dac23860, bd33c8d9, 1f72fc12, 928356e5, e73ea4cf, 6ef96c44, 7a3921bc, 1b2

### VM Aggregation - get node load - use for comparison

In [23]:
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        SUM(power_clean) AS total_power,
        COUNT(*) AS vm_count
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬────────────────────┬──────────┐
│ node_name │    total_power     │ vm_count │
│  varchar  │       double       │  int64   │
├───────────┼────────────────────┼──────────┤
│ 6fdf4d0d  │              187.2 │        1 │
│ 1d247e53  │               0.07 │        7 │
│ a57bfdd7  │              52.27 │        1 │
│ 6ff55332  │               0.24 │        3 │
│ 7f384201  │               0.22 │        4 │
│ 3176c89e  │               0.18 │        3 │
│ e4362c19  │               4.02 │       16 │
│ 8d5a6c7e  │               0.41 │        1 │
│ 13a6f4c8  │                0.1 │        5 │
│ 43e593b3  │               2.99 │        7 │
│    ·      │                 ·  │        · │
│    ·      │                 ·  │        · │
│    ·      │                 ·  │        · │
│ 05c5ef00  │             521.42 │        1 │
│ 8621bb59  │                0.0 │        1 │
│ 417242bc  │               0.09 │        1 │
│ a26788f6  │ 1.2000000000000002 │        9 │
│ bb1bcaff  │               0.21 │

# NODE_t

In [24]:
NODES = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/full_nodes_featurestwo.parquet"

In [25]:
con.execute(f"""
    CREATE TABLE nodes AS
    SELECT * FROM read_parquet('{NODES}')
""")

con.query("""SELECT * FROM nodes LIMIT 5""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (1.2 GiB/1.2 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

## Checking for matching timestamps

# how long is the vm dictionary, and how long is the host dictionary

In [ ]:
timestamps = con.execute("""
    SELECT DISTINCT timestamp FROM vm_final ORDER BY timestamp
""").fetchall()

for (t,) in timestamps:

    # VM state
    vm_df_t = con.execute("""
        SELECT * FROM vm_final WHERE timestamp = ?
    """, [t]).df()

    # Node aggregation
    node_df_t = con.execute("""
        SELECT 
            node_name,
            SUM(power_clean) AS total_power,
            COUNT(*) AS vm_count
        FROM vm_final
        WHERE timestamp = ?
        GROUP BY node_name
    """, [t]).df()

    # → apply your consolidation logic